# RDF & Turtle with rdflib

**Domain:** Symbolic AI & Logic  ·  **recommended addition**  ·  **runnable:** yes

A working refresher on modelling knowledge as a graph of triples, writing it in **Turtle**,
and manipulating it from Python with **rdflib** — the de-facto Python toolkit for RDF.

## 1. What & Why

**RDF (Resource Description Framework)** is a W3C data model for facts. Every fact is a
**triple**: `(subject, predicate, object)` — "Alice knows Bob", "Bob's name is 'Bob'".
A set of triples *is* a directed, labelled graph: subjects and objects are nodes, predicates
are the edges. There is no fixed schema up front; you just keep asserting triples, and the
graph grows.

**The problem it solves.** Relational tables and JSON documents bake in a shape: columns,
nesting, foreign keys. The moment your data is heterogeneous, sparsely populated, or stitched
together from many sources with different vocabularies, that rigidity hurts. RDF flips it: a
**universal, self-describing data model** where every entity and every relationship is named
by a global **IRI** (a URI), so two datasets authored independently can be merged by simply
taking the union of their triples — shared IRIs line up automatically. This is the backbone of
the **Semantic Web**, **Linked Data**, and most knowledge-graph plumbing.

**Turtle** (`.ttl`) is the human-friendly text syntax for RDF — far more readable than RDF/XML,
and the format you'll actually read and write by hand.

**rdflib** is the mature Python library that gives you an in-memory `Graph`, parsers and
serializers for every RDF syntax (Turtle, N-Triples, JSON-LD, RDF/XML…), a full **SPARQL 1.1**
query engine, and pluggable persistent stores.

**Reach for it when** you need to: consume Linked Open Data (DBpedia, Wikidata, schema.org);
integrate messy data from many vocabularies; build a knowledge graph with formal semantics
(pair it with RDFS/OWL — see [`owl`](owl.ipynb)); or prototype SPARQL locally without standing
up a triplestore. **Skip it when** your data is uniform and high-volume (use a relational DB
or a property-graph store like Neo4j), or when you only need a quick lookup table.

## 2. Mental Model

Think of one giant **set of sentences**, each exactly three words long:

```
            predicate
   subject ───────────▶ object
```

- A **node** is either an **IRI** (a global name, `<http://example.org/alice>`), a **literal**
  (a value: `"Alice"`, `30`, `2024-01-01`), or a **blank node** (an anonymous "something").
- Subjects are IRIs or blank nodes; predicates are always IRIs; objects can be any of the three.
- The whole dataset is just the **union of all triples**. Merging two graphs = set union.
  Nodes that share an IRI are *the same node*, even across files and across the web.

So a graph is not a clever index over your data — the triples *are* the data. Querying
(SPARQL) is **graph pattern matching**: you write a little template graph with `?variables`
and ask "where does this shape occur?".

Turtle is just shorthand for writing those triples: `;` reuses the subject, `,` reuses subject
+predicate, `a` is sugar for `rdf:type`, and `@prefix` lets you abbreviate long IRIs.

## 3. Key Concepts

- **Triple / statement** — `(subject, predicate, object)`; the atom of RDF.
- **IRI / URIRef** — a global identifier for a thing or a relation. Reuse across datasets is
  what makes data link up. In rdflib: `URIRef("http://example.org/alice")`.
- **Literal** — a typed value: a lexical string plus an optional **datatype** (`xsd:integer`,
  `xsd:date`, …) **or** a **language tag** (`"chat"@fr`). `Literal(30, datatype=XSD.integer)`.
- **Blank node (bnode)** — a node with no IRI, scoped to one graph; used for structure you don't
  need to name globally (e.g. an address with no canonical URI).
- **Namespace / prefix** — a base IRI you bind to a short prefix (`foaf:` →
  `http://xmlns.com/foaf/0.1/`) so you write `foaf:name` instead of the full IRI.
- **Vocabulary / ontology** — an agreed set of IRIs: **RDF**, **RDFS** (classes, subclass,
  domain/range), **OWL** (richer logic), plus domain vocabularies like **FOAF**, **schema.org**,
  **Dublin Core**. rdflib ships many as importable namespace objects.
- **Graph** — rdflib's container of triples; iterable, set-like (`add`, `remove`, `__len__`,
  `in`), and the thing you parse into and query.
- **SPARQL** — the query language for RDF; `SELECT`, `CONSTRUCT`, `ASK`, `DESCRIBE`. See the
  dedicated [`sparql`](sparql.ipynb) notebook; here we run it via `Graph.query`.
- **Serialization formats** — Turtle, N-Triples (one triple per line, no abbreviation, great
  for diffing/streaming), JSON-LD, RDF/XML, N-Quads (triples + a named-graph column).

## 4. Setup

Pure-Python, no system dependencies. `pip install rdflib` is enough; JSON-LD support is built
into modern rdflib (7.x).

In [ ]:
# Install rdflib if it's missing (no-op when already present).
try:
    import rdflib
except ImportError:
    %pip install -q rdflib
    import rdflib

print("rdflib", rdflib.__version__)

## 5. Worked Examples

### Example 1 — Build a graph triple by triple

We create FOAF-style facts about two people and iterate the graph. Note how a literal carries a
datatype, and how `rdflib.namespace.FOAF` gives us the vocabulary IRIs for free.

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF
from rdflib.namespace import FOAF, XSD

g = Graph()
EX = Namespace("http://example.org/")
g.bind("ex", EX)          # so serializations use the short prefix
g.bind("foaf", FOAF)

alice, bob = EX.alice, EX.bob   # Namespace attribute access -> URIRef

g.add((alice, RDF.type, FOAF.Person))
g.add((alice, FOAF.name, Literal("Alice")))
g.add((alice, FOAF.age, Literal(30, datatype=XSD.integer)))
g.add((alice, FOAF.knows, bob))
g.add((bob, RDF.type, FOAF.Person))
g.add((bob, FOAF.name, Literal("Bob")))

print(len(g), "triples\n")
for s, p, o in g:
    print(f"{s.n3(g.namespace_manager):<12} {p.n3(g.namespace_manager):<11} {o.n3(g.namespace_manager)}")

### Example 2 — Parse Turtle, then query with SPARQL

Turtle in, triples merged into the same graph. Then a `SELECT` with an `OPTIONAL` so people
without an age still show up — the classic RDF "missing data is just an absent triple" pattern.

In [ ]:
turtle = '''
@prefix ex:   <http://example.org/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .

ex:carol a foaf:Person ;
    foaf:name "Carol" ;
    foaf:age  25 ;
    foaf:knows ex:alice .
'''

g.parse(data=turtle, format="turtle")
print(len(g), "triples after merge\n")

q = '''
PREFIX foaf: <http://xmlns.com/foaf/0.1/>
SELECT ?name ?age WHERE {
    ?p a foaf:Person ;
       foaf:name ?name .
    OPTIONAL { ?p foaf:age ?age }
}
ORDER BY ?name
'''
for row in g.query(q):
    print(f"{str(row.name):<8} age={row.age}")

### Example 3 — Serialize back out

The same graph, written as Turtle. Swap `format=` for `"json-ld"`, `"nt"` (N-Triples), or
`"xml"` (RDF/XML). Turtle and N-Triples are the ones you'll reach for most.

In [ ]:
print(g.serialize(format="turtle"))

### Example 4 — Fetching live RDF (network-gated)

rdflib can `parse()` straight from a URL, which is how you pull in Linked Data. That needs the
network, so it's gated behind an env var to keep this notebook executable offline — the call
shape is shown either way.

In [ ]:
import os

if os.getenv("FETCH_REMOTE"):
    remote = Graph()
    remote.parse("https://www.w3.org/People/Berners-Lee/card", format="xml")
    print(len(remote), "triples fetched")
else:
    print("Set FETCH_REMOTE=1 to fetch live RDF. Call shape:")
    print('    Graph().parse("https://www.w3.org/People/Berners-Lee/card", format="xml")')

## 6. Gotchas & Pitfalls

- **`http://` vs `https://` IRIs are different nodes.** IRIs match by exact string. A common
  source of empty query results is the data using one scheme and your query the other, or a
  trailing slash mismatch (`schema.org/name` ≠ `schema.org/name/`).
- **Literals vs IRIs.** `Literal("http://example.org/alice")` is the *string*, not the resource
  `URIRef("http://example.org/alice")`. Mixing them up silently breaks joins. In Turtle, quotes
  mean literal; angle brackets / prefixed names mean IRI.
- **Datatype and language affect equality.** `Literal("30")` (a plain string) ≠
  `Literal(30, datatype=XSD.integer)`, and `"chat"@en` ≠ `"chat"@fr` ≠ `"chat"`. Be deliberate
  about datatypes when you compare or filter.
- **Blank node identity is graph-local and non-portable.** Their internal ids are not stable
  across parses or serializations; never store or rely on a bnode id as a key.
- **`Graph` is a set, so duplicate triples are silently ignored** and there is no inherent
  ordering — always `ORDER BY` in SPARQL if you need determinism.
- **No reasoning by default.** rdflib stores what you assert; it does *not* infer
  `rdfs:subClassOf` or transitive `owl` consequences on its own. You need a reasoner / the
  `owlrl` package, or property-path queries, to get entailments.
- **The default `Graph()` is in-memory.** It vanishes when the process ends. For persistence use
  a store backend (`Graph(store="...")`) or serialize to a file and re-parse.
- **`parse()` guesses format from the file extension, not always correctly.** Pass `format=`
  explicitly for strings and ambiguous sources; `data=` parses a string, `source=`/`location=`
  parses a path or URL.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs rdflib |
|---|---|---|
| **rdflib (RDF graph in Python)** | Linked Data, multi-vocabulary integration, formal semantics, local SPARQL prototyping | In-memory default; not built for millions of triples or high write throughput |
| **Dedicated triplestore** (Apache Jena/Fuseki, GraphDB, Blazegraph, Virtuoso) | Large RDF datasets, persistent SPARQL endpoints, reasoning at scale | Operational overhead; you talk to it over HTTP/SPARQL rather than in-process |
| **Property graph** (Neo4j + Cypher, Memgraph) | Richly-attributed edges, path/traversal-heavy analytics, dev ergonomics | No global IRIs or open-world standards; harder to merge external datasets |
| **Relational DB** (Postgres) | Uniform, high-volume, transactional, well-known schema | Rigid schema; awkward for sparse/heterogeneous facts and many-to-many semantics |
| **Document store** (MongoDB) / **plain JSON** | Self-contained nested records, simple apps | No cross-document identity or graph queries; integration is manual |

**Rule of thumb:** choose RDF/rdflib when *interoperability and meaning* matter — open vocabularies,
data you'll merge with others, or facts that need formal semantics. For closed, high-volume,
schema-stable data, a relational or property-graph store will be simpler and faster. rdflib is
also the perfect place to *learn and prototype* SPARQL before pointing it at a real triplestore.
Related notebooks: [`sparql`](sparql.ipynb), [`owl`](owl.ipynb), [`knowledge-graphs`](knowledge-graphs.ipynb).

## 8. Resources

- **rdflib documentation** — https://rdflib.readthedocs.io/ (start with the "Getting started"
  and "Navigating Graphs" guides).
- **RDF 1.1 Primer (W3C)** — https://www.w3.org/TR/rdf11-primer/ — the gentle, authoritative
  intro to the data model.
- **RDF 1.1 Turtle (W3C)** — https://www.w3.org/TR/turtle/ — the full Turtle grammar and sugar.
- **SPARQL 1.1 Query Language (W3C)** — https://www.w3.org/TR/sparql11-query/ — the query spec.
- **Linked Data: Evolving the Web into a Global Data Space** (Heath & Bizer) —
  http://linkeddatabook.com/editions/1.0/ — free book on Linked Data principles.

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
def parse_turtle(text):
    """Expand the abbreviated form into the set of triples it stands for."""
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE